# 🧠 Universal MRI Skull-Strip + Crop-to-Mask

This notebook takes a **whole-head MRI scan** and a **mask**, then:

1. **Skull-strips** the scan by zeroing out everything outside the mask.
2. **Crops** the result tightly to the mask's bounding box.
3. Gives you back a **NIfTI** (`.nii.gz`) file to download.

It is built to be *universal*: if the mask and scan don't share the exact same
grid (shape / affine), the mask is automatically resampled onto the scan grid
(nearest-neighbor) before being applied. Spatial position (the affine) is kept
correct after cropping.

### How to use
1. Run the cells in order (Runtime ▸ Run all, or ▶️ each cell).
2. When prompted, **drag & drop** (or browse to) your scan, then your mask.
3. The output downloads automatically at the end.

> Both files must be NIfTI (`.nii` or `.nii.gz`).

## 1. Install dependencies

In [ ]:
# nibabel for NIfTI I/O. numpy / scipy / matplotlib are preinstalled on Colab.
!pip install -q nibabel

## 2. Imports, settings & helper functions

Tweak the two settings below if you like:
- `PADDING` &mdash; extra voxels to leave around the mask bounding box (`0` = crop exactly to the mask).
- `MASK_THRESHOLD` &mdash; voxels with a mask value **greater than** this are kept (`0` works for typical binary masks).

In [ ]:
import os
import numpy as np
import nibabel as nib
from nibabel.processing import resample_from_to

# ---------------- Settings ----------------
PADDING = 0          # extra voxels around the mask bounding box (0 = exact crop)
MASK_THRESHOLD = 0   # keep voxels where mask > MASK_THRESHOLD
# ------------------------------------------


def same_grid(img_a, img_b, tol=1e-3):
    """True if two images share shape (first 3 dims) and affine within tolerance."""
    return (img_a.shape[:3] == img_b.shape[:3]) and np.allclose(img_a.affine, img_b.affine, atol=tol)


def get_mask_in_scan_space(scan_img, mask_img):
    """Return mask data on the scan's voxel grid, resampling if necessary."""
    if same_grid(scan_img, mask_img):
        print("✓ Mask and scan share the same grid — no resampling needed.")
        return np.asarray(mask_img.dataobj)
    print("↻ Mask and scan differ in shape/affine — resampling mask onto the scan grid (nearest-neighbor).")
    resampled = resample_from_to(mask_img, (scan_img.shape[:3], scan_img.affine), order=0)
    return np.asarray(resampled.dataobj)


def binarize(mask_data, threshold=0):
    return (mask_data > threshold).astype(np.uint8)


def apply_mask(scan_data, mask_bin):
    """Zero out voxels outside the mask (handles 3D and 4D scans)."""
    if scan_data.ndim == 4:
        out = scan_data * mask_bin[..., None]
    else:
        out = scan_data * mask_bin
    return out.astype(scan_data.dtype)


def crop_to_mask(data, mask_bin, affine, padding=0):
    """Crop `data` to the bounding box of `mask_bin` and return a corrected affine."""
    coords = np.array(np.where(mask_bin > 0))
    if coords.size == 0:
        raise ValueError("The mask is empty after thresholding — nothing to crop to.")
    min_idx = coords.min(axis=1)
    max_idx = coords.max(axis=1)
    min_idx = np.maximum(min_idx - padding, 0)
    max_idx = np.minimum(max_idx + padding, np.array(mask_bin.shape) - 1)
    slc = tuple(slice(int(min_idx[d]), int(max_idx[d]) + 1) for d in range(3))
    cropped = data[slc]
    # Shift the affine origin so the cropped volume stays in the same world space.
    new_affine = affine.copy()
    new_affine[:3, 3] = affine[:3, :3] @ min_idx + affine[:3, 3]
    return cropped, new_affine, (min_idx, max_idx)


print("Helpers ready.")

## 3. Upload the whole-head MRI scan
Drag & drop your scan onto the upload box that appears (or click to browse).

In [ ]:
from google.colab import files

print("⬆️  Upload your WHOLE-HEAD MRI scan (.nii or .nii.gz):")
scan_upload = files.upload()
scan_path = list(scan_upload.keys())[0]
print("Loaded scan:", scan_path)

## 4. Upload the mask
Drag & drop your mask file next.

In [ ]:
print("⬆️  Upload your MASK (.nii or .nii.gz):")
mask_upload = files.upload()
mask_path = list(mask_upload.keys())[0]
print("Loaded mask:", mask_path)

## 5. Skull-strip + crop

In [ ]:
scan_img = nib.load(scan_path)
mask_img = nib.load(mask_path)

print("Scan shape:", scan_img.shape, "| dtype:", scan_img.get_data_dtype())
print("Mask shape:", mask_img.shape, "| dtype:", mask_img.get_data_dtype())

scan_data = np.asarray(scan_img.dataobj)

# Put the mask on the scan's grid, then binarize it.
mask_data = get_mask_in_scan_space(scan_img, mask_img)
mask_bin = binarize(mask_data, threshold=MASK_THRESHOLD)
print("Mask voxels kept:", int(mask_bin.sum()))

# Skull-strip: everything outside the mask becomes 0.
stripped = apply_mask(scan_data, mask_bin)

# Crop tightly to the mask bounding box.
cropped, new_affine, (mn, mx) = crop_to_mask(stripped, mask_bin, scan_img.affine, padding=PADDING)
print("Bounding box (voxel min → max):", mn.tolist(), "→", mx.tolist())
print("Output shape:", cropped.shape)

# Build a clean output image (clean header avoids any scaling surprises).
out_img = nib.Nifti1Image(cropped, new_affine)
out_img.set_data_dtype(cropped.dtype)

# Output filename based on the scan name.
base = os.path.basename(scan_path)
for ext in (".nii.gz", ".nii"):
    if base.endswith(ext):
        base = base[:-len(ext)]
        break
out_name = base + "_skullstripped_cropped_final_trim.nii.gz"
nib.save(out_img, out_name)
print("✅ Saved:", out_name)

## 6. Quick visual check (optional)
A middle axial slice of the result, just to confirm it looks right.

In [ ]:
import matplotlib.pyplot as plt

vol = cropped if cropped.ndim == 3 else cropped[..., 0]
z = vol.shape[2] // 2
plt.figure(figsize=(5, 5))
plt.imshow(np.rot90(vol[:, :, z]), cmap="gray")
plt.title("Skull-stripped & cropped — middle axial slice")
plt.axis("off")
plt.show()

## 7. Download the result

In [ ]:
from google.colab import files
files.download(out_name)